# aira — AI Training on Google Colab

This notebook trains the aira AI model using Google Colab's GPU.
**Run cells in order from Cell 1 using `Shift+Enter`.**

> For detailed explanations, refer to `docs/lessons_JP/06_AI_Mode.md` (Section 4) or `docs/lessons_EN/06_AI_Mode.md`.

---

## ✅ Before opening this notebook, verify the following

Check that the following are ready in your Google Drive "My Drive":

- [ ] Created the folder `My Drive/virtual-robot-race/` (**folder name must be exactly `virtual-robot-race`**)
- [ ] Uploaded `Robot1/model.py` to `My Drive/virtual-robot-race/model.py`
- [ ] Set Runtime → Change runtime type → **GPU**

> ⚠️ **You will upload the training data (training_data/) before Cell 4.**
> After mounting Google Drive in Cell 1, upload the `Robot1/training_data/` folder
> from your local PC via the Google Drive browser tab.

---

## What this notebook does

| Cell | What it does |
|------|-------------|
| Cell 1 | Mount Google Drive and configure paths |
| Cell 2 | Verify PyTorch and CUDA |
| Cell 3 | Create a new iteration folder for this training run |
| ⬆️ Here | **Upload training_data/ to Google Drive** |
| Cell 4 | Copy run folders into the iteration's data_sources/ |
| Cell 5 | Generate dataset manifest (sample counts, completion status) |
| Cell 6 | Set up model, DataLoader, and Data Augmentation |
| Cell 7 | Run training loop (up to 100 epochs, with early stopping) |
| Cell 8 | Visualize training curves |
| Final step | Download `model_best.pth` and place it in local `Robot1/models/` |

---
## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set project root paths
PROJECT_ROOT = "/content/drive/MyDrive/virtual-robot-race"
TRAINING_DATA_ROOT = os.path.join(PROJECT_ROOT, "training_data")
EXPERIMENTS_ROOT = os.path.join(PROJECT_ROOT, "experiments")
ITERATIONS_ROOT = os.path.join(EXPERIMENTS_ROOT, "iterations")

# Verify training_data folder exists
if not os.path.exists(TRAINING_DATA_ROOT):
    print(f"⚠️ WARNING: {TRAINING_DATA_ROOT} not found.")
    print(f"   Upload Robot1/training_data/ to Google Drive before running Cell 4.")
else:
    run_folders = [f for f in os.listdir(TRAINING_DATA_ROOT) if f.startswith('run_')]
    print(f"✓ Training data found: {len(run_folders)} run folders")

# Create experiments/iterations folder
os.makedirs(ITERATIONS_ROOT, exist_ok=True)
print(f"\n✓ Project root: {PROJECT_ROOT}")
print(f"✓ Experiments root: {EXPERIMENTS_ROOT}")

---
## Cell 2: Install libraries

In [ ]:
import sys
import torch
import torchvision

# Verify pre-installed libraries
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

# Verify CUDA
if torch.cuda.is_available():
    print(f"\n✓ CUDA available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ WARNING: CUDA not available. Go to Runtime → Change runtime type → GPU")

# Install additional packages here if needed
# !pip install [package_name]

---
## Cell 3: Create iteration folder

In [ ]:
from datetime import datetime
from pathlib import Path

# Generate iteration name (same timestamp format as local)
timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
iteration_name = f"iteration_{timestamp}"
iteration_path = Path(ITERATIONS_ROOT) / iteration_name

# Create folder structure
iteration_path.mkdir(parents=True, exist_ok=True)
data_sources_path = iteration_path / "data_sources"
data_sources_path.mkdir(exist_ok=True)
plots_path = iteration_path / "plots"
plots_path.mkdir(exist_ok=True)

print(f"✓ Created iteration: {iteration_name}")
print(f"  Path: {iteration_path}")
print(f"\n📋 Next: upload training_data/ to Google Drive if not done yet, then run Cell 4.")

---

## ⬆️ Upload your training data now (before running Cell 4)

**If you have not yet uploaded your training data, do it here.**

1. Open [drive.google.com](https://drive.google.com) in a new browser tab
2. Navigate to `My Drive/virtual-robot-race/`
3. Upload the `Robot1/training_data/` folder from your local PC (drag and drop or use the upload button)
   - You should see folders like `run_20260502_063425/` appear in Google Drive

> 💡 **Already uploaded? Skip this step and run Cell 4 directly.**

---
## Cell 4: Automatically copy all training data

Automatically copies all available `run_` folders to `data_sources/`.

In [ ]:
import shutil
import json

# List available run_ folders
available_runs = sorted([f for f in os.listdir(TRAINING_DATA_ROOT) if f.startswith('run_')])

print(f"Available run_ folders ({len(available_runs)} total):")
print("-" * 80)
for i, run_folder in enumerate(available_runs, 1):
    run_path = Path(TRAINING_DATA_ROOT) / run_folder

    # Count samples in metadata.csv
    csv_file = run_path / "metadata.csv"
    if csv_file.exists():
        with open(csv_file, 'r') as f:
            num_lines = sum(1 for _ in f) - 1  # exclude header row
        print(f"{i:2d}. {run_folder} ({num_lines} samples)")
    else:
        print(f"{i:2d}. {run_folder} (metadata.csv not found)")

print("\n" + "=" * 80)
print(f"📂 Copying all {len(available_runs)} run_ folders to data_sources/")
print("=" * 80)

# Copy all runs to data_sources/
selected_runs = available_runs

print(f"\n📂 Copying {len(selected_runs)} run_ folder(s)...")
for run_folder in selected_runs:
    src = Path(TRAINING_DATA_ROOT) / run_folder
    dst = data_sources_path / run_folder

    if dst.exists():
        print(f"  ⏭️  {run_folder} (already exists, skipping)")
    else:
        shutil.copytree(src, dst)
        print(f"  ✓ {run_folder}")

print(f"\n✓ Copy complete: {data_sources_path}")
print(f"  {len(selected_runs)} run_ folder(s) total")

---
## Cell 5: Generate dataset manifest

Analyzes data in `data_sources/` and generates `dataset_manifest.json`.

In [ ]:
import csv
import json
from pathlib import Path
from datetime import datetime

def create_dataset_manifest(data_sources_path, output_path):
    """
    Analyze run_ folders in data_sources/ and generate dataset_manifest.json.
    """
    manifest = {
        "created_at": datetime.now().isoformat(),
        "runs": [],
        "total_samples": 0,
        "completed_runs": 0,
        "incomplete_runs": 0
    }

    run_folders = sorted([f for f in data_sources_path.iterdir() if f.name.startswith('run_')])

    print(f"Analyzing {len(run_folders)} run folders...\n")

    for run_folder in run_folders:
        csv_file = run_folder / "metadata.csv"

        if not csv_file.exists():
            print(f"⚠️  {run_folder.name}: metadata.csv not found, skipping")
            continue

        # Load CSV data
        with open(csv_file, 'r') as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        if len(rows) == 0:
            print(f"⚠️  {run_folder.name}: Empty CSV, skipping")
            continue

        # Filter racing frames (Lap0, Lap1, Lap2, ...)
        # Excludes: StartSequence, Finish, Fallen, FalseStart
        racing_rows = [
            row for row in rows
            if row.get('status', '').startswith('Lap')
        ]

        if len(racing_rows) == 0:
            print(f"⚠️  {run_folder.name}: No Lap frames found, skipping")
            continue

        # Determine completion status
        # Unity spec: Lap0=lap1 in progress, Lap1=lap1 done (lap2 in progress), Lap2=2-lap finish
        all_statuses = set(row.get('status') for row in racing_rows)
        max_lap = max([int(s.replace('Lap', '')) for s in all_statuses if s.startswith('Lap')], default=0)
        race_completed = (max_lap >= 1)  # Lap1 or higher = 2-lap finish

        # Race time
        race_time_ms = float(racing_rows[-1].get('race_time_ms', 0))
        race_time_sec = race_time_ms / 1000.0

        # Drive/steer statistics
        torques = [float(row['drive_torque']) for row in racing_rows]
        steers = [float(row['steer_angle']) for row in racing_rows]
        avg_torque = sum(torques) / len(torques)
        avg_steer = sum(steers) / len(steers)

        run_info = {
            "run_id": run_folder.name,
            "num_samples": len(racing_rows),
            "race_completed": race_completed,
            "max_lap": max_lap,
            "race_time_sec": race_time_sec,
            "avg_torque": round(avg_torque, 4),
            "avg_steer": round(avg_steer, 4)
        }

        manifest["runs"].append(run_info)
        manifest["total_samples"] += len(racing_rows)

        if race_completed:
            manifest["completed_runs"] += 1
            status = f"✓ COMPLETED (Lap{max_lap})"
        else:
            manifest["incomplete_runs"] += 1
            status = f"⚠️ INCOMPLETE (Lap{max_lap})"

        print(f"{status} {run_folder.name}: {len(racing_rows)} samples, {race_time_sec:.1f}s")

    # Save manifest
    with open(output_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    return manifest

# Generate manifest
manifest_path = iteration_path / "dataset_manifest.json"
manifest = create_dataset_manifest(data_sources_path, manifest_path)

print("\n" + "=" * 80)
print("📊 Dataset Summary")
print("=" * 80)
print(f"Total runs: {len(manifest['runs'])}")
print(f"  Completed: {manifest['completed_runs']}")
print(f"  Incomplete: {manifest['incomplete_runs']}")
print(f"Total samples: {manifest['total_samples']:,}")
print(f"\n✓ Manifest saved: {manifest_path}")

---
## Cell 6: Model definition and dataset preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import csv
from pathlib import Path
import sys
import random
import numpy as np

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# ===========================
# Seed (for reproducibility)
# ===========================
def set_seed(seed=42):
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Random seed: {SEED}")

# ===========================
# Import model.py
# ===========================
MODEL_FILE = Path(PROJECT_ROOT) / "model.py"

if not MODEL_FILE.exists():
    print(f"⚠️ ERROR: {MODEL_FILE} not found.")
    print(f"   Upload Robot1/model.py to Google Drive:")
    print(f"   Destination: /MyDrive/virtual-robot-race/model.py")
    raise FileNotFoundError(f"model.py not found at {MODEL_FILE}")

# Add model directory to Python path
sys.path.insert(0, str(MODEL_FILE.parent))

# Import DrivingNetwork from model.py
from model import DrivingNetwork

print(f"✓ model.py imported from: {MODEL_FILE}")

# ===========================
# Dataset (same filtering as local train.py)
# ===========================
class RobotRaceDataset(Dataset):
    # Use same status filter as train.py: Lap0, Lap1, Lap2, Finish
    VALID_RACING_STATUS = ["Lap0", "Lap1", "Lap2", "Finish"]

    def __init__(self, data_sources_path, transform=None, exclude_start_sequence=True):
        self.data_sources_path = Path(data_sources_path)
        self.transform = transform
        self.samples = []

        run_folders = sorted([f for f in self.data_sources_path.iterdir() if f.name.startswith('run_')])

        print(f"\nLoading dataset from {len(run_folders)} run folders...")
        print("-" * 80)

        total_loaded = 0
        total_skipped = 0

        for run_folder in run_folders:
            csv_file = run_folder / "metadata.csv"
            if not csv_file.exists():
                print(f"⚠️  {run_folder.name}: metadata.csv not found, skipping")
                continue

            loaded_count = 0
            skipped_count = 0
            skipped_reasons = {}

            with open(csv_file, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    status = row.get('status', '')

                    # Filter: use racing frames only
                    if exclude_start_sequence and status not in self.VALID_RACING_STATUS:
                        skipped_count += 1
                        skipped_reasons[status] = skipped_reasons.get(status, 0) + 1
                        continue

                    # Get image path from filename column
                    filename = row['filename']
                    image_path = run_folder / "images" / filename

                    if image_path.exists():
                        self.samples.append({
                            'image_path': str(image_path),
                            'drive_torque': float(row['drive_torque']),
                            'steer_angle': float(row['steer_angle']),
                            'soc': float(row['soc'])
                        })
                        loaded_count += 1
                    else:
                        skipped_count += 1
                        skipped_reasons['ImageNotFound'] = skipped_reasons.get('ImageNotFound', 0) + 1

            total_loaded += loaded_count
            total_skipped += skipped_count

            if skipped_count > 0:
                skip_details = ", ".join([f"{k}:{v}" for k, v in skipped_reasons.items()])
                print(f"  ✓ {run_folder.name}: {loaded_count} samples (skipped {skipped_count}: {skip_details})")
            else:
                print(f"  ✓ {run_folder.name}: {loaded_count} samples")

        print("-" * 80)
        print(f"✓ Dataset loaded: {total_loaded} samples total")
        if total_skipped > 0:
            print(f"  Skipped: {total_skipped} frames (non-racing or missing image)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = Image.open(sample['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)

        soc = torch.tensor([sample['soc']], dtype=torch.float32)

        # Target order matches model.py output: [drive_torque, steer_angle]
        target = torch.tensor([sample['drive_torque'], sample['steer_angle']], dtype=torch.float32)

        return image, soc, target

# ===========================
# Data transforms (Augmentation for training, plain for validation)
# ===========================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ===========================
# Create dataset and split
# ===========================
full_dataset = RobotRaceDataset(data_sources_path, transform=train_transform)

# Train/Val split (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Note: random_split returns Subset objects that share the parent dataset's transform.
# Validation augmentation is intentionally omitted for a fair evaluation.

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# ===========================
# Create DataLoaders
# ===========================
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# ===========================
# Model, loss, optimizer
# ===========================
model = DrivingNetwork().to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.0001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
)

print(f"\n✓ Model initialized on {device}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Cell 7: Train model

Training loop with early stopping

In [ ]:
import time
from tqdm import tqdm

# ===========================
# Training parameters
# ===========================
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 0.0001

# ===========================
# Early Stopping
# ===========================
class EarlyStopping:
    """Early stopping to prevent overfitting."""

    def __init__(self, patience=15, min_delta=0.0001, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False
        self.best_epoch = 0

    def __call__(self, val_loss, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_epoch = epoch
            return False
        else:
            self.counter += 1
            if self.verbose and self.counter > 0:
                print(f"      [EarlyStopping] No improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.should_stop = True
                if self.verbose:
                    print(f"      [EarlyStopping] Stopping! Best epoch was {self.best_epoch + 1}")
                return True
        return False

# ===========================
# Training Logger (CSV)
# ===========================
class TrainingLogger:
    """Log training metrics to CSV file."""

    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.metrics_history = []

        with open(self.log_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                'epoch', 'train_loss', 'train_torque_loss', 'train_steer_loss',
                'val_loss', 'val_torque_loss', 'val_steer_loss',
                'learning_rate', 'is_best', 'timestamp'
            ])

    def log(self, epoch, train_metrics, val_metrics, learning_rate, is_best):
        from datetime import datetime

        row = {
            'epoch': epoch + 1,
            'train_loss': train_metrics['loss'],
            'train_torque_loss': train_metrics['torque_loss'],
            'train_steer_loss': train_metrics['steer_loss'],
            'val_loss': val_metrics['loss'],
            'val_torque_loss': val_metrics['torque_loss'],
            'val_steer_loss': val_metrics['steer_loss'],
            'learning_rate': learning_rate,
            'is_best': is_best,
            'timestamp': datetime.now().isoformat()
        }
        self.metrics_history.append(row)

        with open(self.log_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(list(row.values()))

# ===========================
# Training loop
# ===========================
train_losses = []
val_losses = []
train_torque_losses = []
train_steer_losses = []
val_torque_losses = []
val_steer_losses = []
learning_rates = []

best_val_loss = float('inf')
best_model_path = iteration_path / "model_best.pth"
final_model_path = iteration_path / "model.pth"

early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=MIN_DELTA)

log_path = iteration_path / "training_log.csv"
logger = TrainingLogger(log_path)

print("\n" + "=" * 80)
print("Starting training...")
print("=" * 80)

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_torque_loss = 0.0
    train_steer_loss = 0.0

    for images, socs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False):
        images = images.to(device)
        socs = socs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images, socs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        with torch.no_grad():
            torque_loss = nn.MSELoss()(outputs[:, 0], targets[:, 0])
            steer_loss = nn.MSELoss()(outputs[:, 1], targets[:, 1])
            train_torque_loss += torque_loss.item()
            train_steer_loss += steer_loss.item()

    train_loss /= len(train_loader)
    train_torque_loss /= len(train_loader)
    train_steer_loss /= len(train_loader)

    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_torque_loss = 0.0
    val_steer_loss = 0.0

    with torch.no_grad():
        for images, socs, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]  ", leave=False):
            images = images.to(device)
            socs = socs.to(device)
            targets = targets.to(device)

            outputs = model(images, socs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()

            torque_loss = nn.MSELoss()(outputs[:, 0], targets[:, 0])
            steer_loss = nn.MSELoss()(outputs[:, 1], targets[:, 1])
            val_torque_loss += torque_loss.item()
            val_steer_loss += steer_loss.item()

    val_loss /= len(val_loader)
    val_torque_loss /= len(val_loader)
    val_steer_loss /= len(val_loader)

    epoch_time = time.time() - epoch_start

    # --- Update learning rate ---
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)

    # --- Save best model ---
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        torch.save(model.state_dict(), final_model_path)
        best_marker = " ✓ NEW BEST"
    else:
        best_marker = ""

    # --- Log metrics ---
    train_metrics = {'loss': train_loss, 'torque_loss': train_torque_loss, 'steer_loss': train_steer_loss}
    val_metrics = {'loss': val_loss, 'torque_loss': val_torque_loss, 'steer_loss': val_steer_loss}
    logger.log(epoch, train_metrics, val_metrics, current_lr, is_best)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_torque_losses.append(train_torque_loss)
    train_steer_losses.append(train_steer_loss)
    val_torque_losses.append(val_torque_loss)
    val_steer_losses.append(val_steer_loss)
    learning_rates.append(current_lr)

    # --- Print progress ---
    print(
        f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
        f"Train: {train_loss:.6f} (T:{train_torque_loss:.4f} S:{train_steer_loss:.4f}) | "
        f"Val: {val_loss:.6f} (T:{val_torque_loss:.4f} S:{val_steer_loss:.4f}) | "
        f"LR: {current_lr:.2e} | {epoch_time:.1f}s{best_marker}"
    )

    # --- Early stopping check ---
    if early_stopping(val_loss, epoch):
        print(f"\n⏹️  Early stopping triggered at epoch {epoch+1}")
        break

print("=" * 80)

# Verify final model is saved
import shutil
if best_model_path.exists() and not final_model_path.exists():
    shutil.copy(best_model_path, final_model_path)

print("\n✓ Training complete!")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Model saved to: {final_model_path}")
print(f"Training log: {log_path}")
print(f"\n📥 Next step: download model_best.pth and place it in local Robot1/models/model.pth")

---
## Cell 8: Visualize results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ===========================
# Plot training curves
# ===========================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs = range(1, len(train_losses) + 1)

# Total Loss
axes[0, 0].plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(epochs, val_losses, 'r-', label='Val Loss', linewidth=2)
axes[0, 0].axhline(y=best_val_loss, color='g', linestyle='--', label=f'Best Val Loss ({best_val_loss:.6f})', linewidth=1)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('MSE Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Torque Loss
axes[0, 1].plot(epochs, train_torque_losses, 'b-', label='Train Torque Loss', linewidth=2)
axes[0, 1].plot(epochs, val_torque_losses, 'r-', label='Val Torque Loss', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Torque Loss', fontsize=12)
axes[0, 1].set_title('Drive Torque Loss', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Steer Loss
axes[1, 0].plot(epochs, train_steer_losses, 'b-', label='Train Steer Loss', linewidth=2)
axes[1, 0].plot(epochs, val_steer_losses, 'r-', label='Val Steer Loss', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Steer Loss', fontsize=12)
axes[1, 0].set_title('Steer Angle Loss', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs, learning_rates, 'g-', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = plots_path / "training_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print(f"\n✓ Plot saved: {plot_path}")

# ===========================
# Summary
# ===========================
print("\n" + "=" * 80)
print("📊 Training Summary")
print("=" * 80)
print(f"Total epochs: {len(train_losses)}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss: {val_losses[-1]:.6f}")

# Overfitting check
overfitting_gap = val_losses[-1] - train_losses[-1]
print(f"\nOverfitting check:")
if overfitting_gap > 0.01:
    print(f"  ⚠️ Possible overfitting detected (gap={overfitting_gap:.6f})")
else:
    print(f"  ✓ No significant overfitting (gap={overfitting_gap:.6f})")

print(f"\nLearning rate:")
print(f"  Initial: {learning_rates[0]:.2e}")
print(f"  Final: {learning_rates[-1]:.2e}")

print("\n" + "=" * 80)
print("🎉 All done! Next steps:")
print("=" * 80)
print(f"1. Download {final_model_path}")
print(f"2. Copy to local Robot1/models/model.pth")
print(f"3. Run aira in AI mode and test your model")
print("=" * 80)

# ===========================
# Preview training log
# ===========================
if log_path.exists():
    print(f"\n📋 Training log preview (first 5 and last 5 rows):")
    print("-" * 80)
    df = pd.read_csv(log_path)
    print("\nFirst 5 rows:")
    print(df.head().to_string(index=False))
    print("\nLast 5 rows:")
    print(df.tail().to_string(index=False))
    print("-" * 80)